# 6.5. 池化层

通常当我们处理图像时，我们希望逐渐降低隐藏表示的空间分辨率、聚集信息，这样随着我们在神经网络中层叠的上升，每个神经元对其敏感的感受野（输入）就越大。

池化层的主要目的是：
1. **降低卷积层对位置的敏感性**
2. **降低空间维度**（下采样）

In [ ]:
import tensorflow as tf
import numpy as np

## 6.5.1. 最大池化和平均池化

池化层与卷积层类似，都是在输入张量的每个区域上运算。但是，池化层不包含参数，而是计算固定的函数。

In [ ]:
def pool2d(X, pool_size, mode='max'):
    """实现池化层的前向传播"""
    p_h, p_w = pool_size
    Y = tf.Variable(tf.zeros(
        (X.shape[0] - p_h + 1, X.shape[1] - p_w + 1)))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j].assign(tf.reduce_max(X[i: i + p_h, j: j + p_w]))
            elif mode == 'avg':
                Y[i, j].assign(tf.reduce_mean(X[i: i + p_h, j: j + p_w]))
    return Y

In [ ]:
# 验证池化层的输出
X = tf.constant([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
print("输入 X:")
print(X.numpy())
print("\n最大池化结果（2×2窗口）:")
print(pool2d(X, (2, 2)).numpy())
print("\n平均池化结果（2×2窗口）:")
print(pool2d(X, (2, 2), 'avg').numpy())

## 6.5.2. 填充和步幅

与卷积层一样，池化层也可以改变输出形状。我们可以通过填充和步幅来调整输出形状。

In [ ]:
# 构造一个输入张量X，形状为(1, 4, 4, 1)
X = tf.reshape(tf.range(16, dtype=tf.float32), (1, 4, 4, 1))
print("输入 X:")
print(X[0, :, :, 0].numpy())

In [ ]:
# 默认情况下，TensorFlow中步幅与池化窗口的大小相同
pool2d_layer = tf.keras.layers.MaxPool2D(pool_size=[3, 3])
result = pool2d_layer(X)
print(f"3×3池化，默认步幅=3: {X.shape} -> {result.shape}")
print(result[0, :, :, 0].numpy())

In [ ]:
# 手动设置填充和步幅
pool2d_layer = tf.keras.layers.MaxPool2D(
    pool_size=[3, 3], padding='same', strides=2)
result = pool2d_layer(X)
print(f"3×3池化，填充=same，步幅=2: {X.shape} -> {result.shape}")
print(result[0, :, :, 0].numpy())

In [ ]:
# 设置一个任意大小的矩形池化窗口，并分别设置高度和宽度的填充和步幅
pool2d_layer = tf.keras.layers.MaxPool2D(
    pool_size=[2, 3], padding='same', strides=(2, 3))
result = pool2d_layer(X)
print(f"2×3池化，步幅=(2,3): {X.shape} -> {result.shape}")
print(result[0, :, :, 0].numpy())

## 6.5.3. 多个通道

在处理多通道输入数据时，池化层对每个输入通道分别池化，而不是像卷积层那样对通道求和。这意味着池化层的输出通道数与输入通道数相同。

In [ ]:
# 在通道维度上连结张量X和X+1
X = tf.concat([X, X + 1], 3)  # 在通道维度连结
print(f"输入形状（含2个通道）: {X.shape}")

pool2d_layer = tf.keras.layers.MaxPool2D(pool_size=[3, 3], padding='same', strides=2)
result = pool2d_layer(X)
print(f"池化后形状: {result.shape}")
print("\n通道1:")
print(result[0, :, :, 0].numpy())
print("\n通道2:")
print(result[0, :, :, 1].numpy())

## 6.5.4. 实际应用示例

In [ ]:
# 常见的池化使用场景
print("典型CNN架构中的池化应用：\n")

# 输入图像
X = tf.random.uniform((1, 224, 224, 3))
print(f"输入图像: {X.shape}")

# 第一个卷积块
conv1 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')
pool1 = tf.keras.layers.MaxPool2D(2, strides=2)
Y = pool1(conv1(X))
print(f"卷积+池化后: {Y.shape} (尺寸减半)")

# 第二个卷积块
conv2 = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')
pool2 = tf.keras.layers.MaxPool2D(2, strides=2)
Y = pool2(conv2(Y))
print(f"再次卷积+池化: {Y.shape} (再次减半)")

# 全局平均池化
global_pool = tf.keras.layers.GlobalAveragePooling2D()
Y = global_pool(Y)
print(f"全局平均池化: {Y.shape} (空间维度消失)")

## 小结

1. **最大池化**：提取窗口中的最大值，保留最显著的特征
2. **平均池化**：计算窗口中的平均值，平滑特征
3. **通道独立**：池化层对每个通道独立操作，输出通道数=输入通道数
4. **无参数**：池化层没有可学习的参数
5. **作用**：
   - 降低空间维度
   - 增加平移不变性
   - 控制过拟合
   - 减少计算量
6. **常见配置**：2×2窗口，步幅2（将特征图大小减半）